# 01 · Meet the OpenAI Agents SDK

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

The **OpenAI Agents SDK** is a small Python framework for building agents
from a handful of primitives. Despite the name, it isn't tied to OpenAI's
models: it talks to anything with an OpenAI-compatible API, including a local
Ollama model, which is what this whole section uses.

This notebook gives you the mental model, runs your first agent, and shows how
the SDK's philosophy differs from a graph framework like LangGraph.

**You will learn**
- What the SDK is, and its design philosophy: *few primitives, Python for everything else*
- The primitives: **Agent, Runner, tools, handoffs, guardrails, sessions, tracing**
- What the built-in **agent loop** does for you
- Your first agent, running on Ollama, and how to read its result
- When to choose this SDK, and when not to

**Prerequisites:** Python and the idea of an LLM calling a tool. No earlier
phase is needed. *Optional background:* `00_CONCEPTS/` explains orchestration
without any framework.

## 1 · Why this SDK exists

Almost every agent is the same loop: send the conversation to a model, run any
tools it asks for, send the results back, repeat until it answers. Writing that
loop yourself is easy once. Writing it *well* is not: tool schemas, parallel tool
calls, structured final answers, passing control between agents, safety checks,
conversation memory, and a record of what happened.

The Agents SDK packages exactly that, and deliberately **no more**:

| Primitive | What it is |
|---|---|
| **Agent** | a model + instructions + tools (+ optional handoffs, guardrails, output type) |
| **Runner** | runs the agent loop until there's a final answer |
| **Tools** | Python functions (or other agents, or MCP servers) the model can call |
| **Handoffs** | an agent passing control of the conversation to another agent |
| **Guardrails** | checks that run on the input or output and can stop the run |
| **Sessions** | conversation memory across runs |
| **Tracing** | a record of every model call, tool call and handoff |

**Everything else is plain Python.** There's no graph, no state schema, no
special workflow language. If you want steps A then B, you `await` A and then
`await` B. If you want them in parallel, you use `asyncio.gather`. The SDK calls
this orchestrating *via code* (you decide the flow) or *via the LLM* (the agent
decides, using tools and handoffs). You'll do both.

## 2 · How it compares to a graph framework

If you've done the LangGraph section, this table will orient you. If not, skip it.

| | OpenAI Agents SDK | LangGraph |
|---|---|---|
| The unit you build | an **agent** | a **graph** of nodes |
| The agent loop | built in (`Runner`) | you build it (or use `create_agent`) |
| Multi-step workflows | ordinary Python (`await`, `gather`, `if`) | edges, routers, `Send` |
| State | the conversation + a context object you pass in | an explicit, typed state schema |
| Checkpoints after every step | no; you can serialise a run when it pauses for approval | yes, by design |
| Amount of code for a simple agent | very little | more |

The trade-off in one sentence: **the SDK makes agents quick to build and easy
to read, and leaves long-running durability and complex flow control to you.**
Notebooks 08, 09 and the project look at exactly where that line is.

## 3 · Your first agent

Every notebook in this section starts with the same setup cell: find this
section's folder so `common.py` can be imported. `get_model()` returns the model
to use: by default, Ollama's `qwen3-coder:30b` through its OpenAI-compatible API.
(Notebook 03 explains exactly how.)

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

from agents import Agent, Runner
from common import get_model

model = get_model()
print(type(model).__name__, '→', model.model)

An agent is a **configuration object**: a name, instructions (its system prompt)
and a model. Creating it does nothing yet:

In [ ]:
tutor = Agent(
    name='Tutor',
    instructions='You explain technical ideas to students in two short sentences.',
    model=model,
)

result = await Runner.run(tutor, 'What is an AI agent?')
print(result.final_output)

`Runner.run` is **async**, so we `await` it. Jupyter supports `await` at the top
level. In a normal script you'd write `asyncio.run(main())`, or use
`Runner.run_sync(...)`, which does that for you. (`run_sync` *doesn't* work
inside Jupyter, because a notebook already has an event loop running.)

## 4 · Adding a tool, and what the loop does

Now give the agent a tool. `@function_tool` turns a Python function into a tool:
the name, docstring and type hints become the schema the model reads.

In [ ]:
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    fake = {'pune': '31°C, humid', 'oslo': '4°C, rain', 'lima': '19°C, cloudy'}
    return f"{city}: {fake.get(city.lower(), '20°C, clear')}"


travel = Agent(
    name='Travel assistant',
    instructions='Answer travel questions. Use tools for facts. Be brief.',
    model=model,
    tools=[get_weather],
)
result = await Runner.run(travel, 'Should I pack an umbrella for Oslo or for Pune?')
print(result.final_output)

One call to `Runner.run`, but several things happened inside. This is the
**agent loop** the Runner implements:

```
 Runner.run(agent, input)
   │
   ▼
 ┌─► call the model with: instructions + conversation + tool schemas
 │         │
 │         ├── the model returned a FINAL ANSWER ──────────────────► return RunResult
 │         │
 │         ├── the model asked for TOOLS ──► run them (in parallel) ──┐
 │         │                                                          │
 │         └── the model asked for a HANDOFF ──► switch agent ────────┤
 │                                                                    │
 └──────────── add the results to the conversation ◄──────────────────┘
               (stop with MaxTurnsExceeded after max_turns rounds)
```

`result.new_items` records every step of that loop:

In [ ]:
for item in result.new_items:
    kind = type(item).__name__
    if kind == 'ToolCallItem':
        print(f'{kind:<18} {item.raw_item.name}({item.raw_item.arguments})')
    elif kind == 'ToolCallOutputItem':
        print(f'{kind:<18} {item.output}')
    else:
        print(f'{kind:<18} {str(getattr(item.raw_item, "content", ""))[:80]}…')

Two tool calls (one per city), their outputs, then the final message. The
SDK took care of the schemas, the calls, feeding results back and deciding
when the run was finished.

A few other useful parts of the result:

In [ ]:
print('final_output :', result.final_output[:80], '…')
print('last_agent   :', result.last_agent.name)
print('model calls  :', len(result.raw_responses))
print('tokens       :', result.context_wrapper.usage.total_tokens)

## 5 · What this section covers

Every notebook follows the same shape: why, how it works, annotated code,
trade-offs and mistakes, check yourself.

| # | Notebook | You'll be able to… |
|---|---|---|
| 01 | Meet the SDK | explain the primitives and run an agent (this notebook) |
| 02 | Agents and the Runner | configure agents; understand exactly when a run ends; pass context |
| 03 | Models and structured output | use Ollama or OpenAI; get typed results with `output_type`; count tokens |
| 04 | Tools | write good tools; handle tool errors; control what happens after a tool; use agents as tools |
| 05 | Orchestrating in code | chains, routers, parallel agents, and refine loops in plain Python |
| 06 | Handoffs and multi-agent | triage and specialists; filter what the next agent sees; manager pattern |
| 07 | Guardrails | block bad input or output before it does damage |
| 08 | Human-in-the-loop | require approval for tools; pause, save, and resume a run later |
| 09 | Sessions and state | conversation memory; what survives a crash and what doesn't |
| 10 | Streaming, tracing and hooks | stream tokens and events; record traces locally |
| 11 | MCP tools | give agents tools from MCP servers |
| 12 | Production concerns | errors, retries, timeouts, limits, cost, and testing with a fake model |
| ⭐ | Project | the Nimbus Briefing Assistant, built with the SDK |

## 6 · When to choose the SDK

| Choose it when… | Consider something else when… |
|---|---|
| you want working agents with little code | runs last hours and must resume after any crash (LangGraph, Temporal) |
| your flow is simple enough to express as ordinary Python | you need a complex, inspectable workflow with branching and checkpoints (LangGraph) |
| you want handoffs, guardrails and tracing built in | you want role-played teams with minimal code (CrewAI) |
| you may use OpenAI's hosted tools (web search, file search) later | you must avoid any vendor coupling in the API design |

## 7 · Common mistakes (beginner edition)

- **Using `Runner.run_sync` in Jupyter.** It fails because the notebook's event
  loop is already running. Use `await Runner.run(...)`.
- **Leaving tracing on without an OpenAI key.** The SDK tries to upload traces to
  OpenAI by default. `common.py` turns tracing export off unless a key is set.
- **Thinking an `Agent` runs by itself.** It's configuration; only `Runner` runs it.
- **Reading `result.final_output` as a string every time.** With an `output_type`
  (notebook 03) it's a typed object.

## 8 · Check yourself

<details><summary><b>1.</b> Name the SDK's primitives and say which one actually executes an agent.</summary>

Agent, Runner, tools, handoffs, guardrails, sessions, tracing. The Runner executes: it runs the agent loop until there's a final output.
</details>

<details><summary><b>2.</b> How does the SDK express "run step A, then step B, and C and D in parallel"?</summary>

In plain Python: `await` A, then `await` B, and `asyncio.gather(C, D)` for the parallel part. The SDK has no graph language; orchestration in code is ordinary async Python.
</details>

<details><summary><b>3.</b> What ends the agent loop?</summary>

A final output from the model (a message with no tool calls, or a valid `output_type` object), or an error such as `MaxTurnsExceeded`. Notebook 02 gives the exact rules.
</details>

<details><summary><b>4.</b> What's the main trade-off compared with a graph framework?</summary>

Much less code and a simple mental model, in exchange for fewer built-in guarantees: no per-step checkpoints, and flow control is your own Python. Durability for long runs is up to you.
</details>

## Takeaway

The Agents SDK is **a few primitives plus Python**. An `Agent` is configuration,
`Runner` runs the loop (model → tools/handoffs → model → … → final output), and
everything around it (sequencing, parallelism, branching) is ordinary async code.

Next → `02_agents_and_the_runner.ipynb`: exactly how the loop decides it's
finished, and how to pass your own data through a run.